# Step 1: Downloading Census Data
To get the data we need the census API Key the code here will ask you for the key then run and get the fields. 
to ge the Census API Key, use this website: https://api.census.gov/data/key_signup.html

In [1]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests

YEAR = 2023
STATE_FIPS = "06"  # California. Keep as a string to preserve the leading zero.
OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)

fields = {
    "B17001_001E": "poverty_universe",
    "B17001_002E": "below_poverty",
    "B01003_001E": "population",
    "B17001_001M": "poverty_universe_moe",
    "B17001_002M": "below_poverty_moe",
    "B01003_001M": "population_moe",
}

api_key = os.getenv("CENSUS_API_KEY") or getpass("Census API key: ")

response = requests.get(
    f"https://api.census.gov/data/{YEAR}/acs/acs5",
    params={
        "get": ",".join(["NAME", *fields]),
        "for": "tract:*",
        "in": f"state:{STATE_FIPS} county:*",
        "key": api_key,
    },
    timeout=90,
)

if not response.ok:
    raise RuntimeError(f"Census request failed with HTTP {response.status_code}.")

try:
    rows = response.json()
except ValueError:
    raise RuntimeError(
        "The Census API did not return data. Check that your API key is active."
    ) from None

tracts = pd.DataFrame(rows[1:], columns=rows[0]).rename(columns=fields)
tracts.head()

,NAME,poverty_universe,below_poverty,population,poverty_universe_moe,below_poverty_moe,population_moe,state,county,tract
0,Census Tract 4001; Alameda County; California,3075,134,3094,451,96,452,06,001,400100
1,Census Tract 4002; Alameda County; California,2093,164,2093,224,88,224,06,001,400200
2,Census Tract 4003; Alameda County; California,5707,310,5727,607,151,607,06,001,400300
3,Census Tract 4004; Alameda County; California,4351,343,4395,636,219,640,06,001,400400
4,Census Tract 4005; Alameda County; California,3816,397,3822,736,209,737,06,001,400500


## Step 1.2 Calculate Poverty and Check the data

In [2]:
# Census geographic identifiers are strings. A tract GEOID has 11 digits.
tracts["county_fips"] = tracts["state"] + tracts["county"]
tracts["tract_geoid"] = tracts["county_fips"] + tracts["tract"]

for col in fields.values():
    values = pd.to_numeric(tracts[col], errors="coerce")
    tracts[col] = values.where(values >= 0)  # Remove negative Census placeholders.

tracts["at_or_above_poverty"] = tracts["poverty_universe"] - tracts["below_poverty"]

valid_denominator = tracts["poverty_universe"].where(tracts["poverty_universe"] > 0)
tracts["poverty_pct"] = 100 * tracts["below_poverty"] / valid_denominator

assert tracts["tract_geoid"].str.fullmatch(r"\d{11}").all()
assert tracts["tract_geoid"].is_unique
assert tracts["poverty_pct"].dropna().between(0, 100).all()

print("Number of census tracts:", len(tracts))
print("Tracts with missing or undefined poverty rate:", tracts["poverty_pct"].isna().sum())

display(
    tracts[
        ["tract_geoid", "NAME", "population", "below_poverty",
         "at_or_above_poverty", "poverty_pct"]
    ].head()
)

tracts.to_parquet(OUT / f"tract_poverty_{STATE_FIPS}_{YEAR}.parquet", index=False)

Number of census tracts: 9129
Tracts with missing or undefined poverty rate: 76


,tract_geoid,NAME,population,below_poverty,at_or_above_poverty,poverty_pct
0,06001400100,Census Tract 4001; Alameda County; California,3094,134,2941,4.357724
1,06001400200,Census Tract 4002; Alameda County; California,2093,164,1929,7.835643
2,06001400300,Census Tract 4003; Alameda County; California,5727,310,5397,5.431926
3,06001400400,Census Tract 4004; Alameda County; California,4395,343,4008,7.883245
4,06001400500,Census Tract 4005; Alameda County; California,3822,397,3419,10.403564


# Step 2 Cleaning Pudl and Merging Pudl Data 
Pudl has given us data from EIA that is already clean but we still need to look at and merge it with the other data that we want to look at that pudl offers. 
Additionally, Pudl hasnt cleaned some of the EIA data so we might need to manually retrieve and clean that ourselves. 

In [2]:
%pip install pandas pyarrow s3fs

  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ------------------ --------------------- 7.3/16.1 MB 35.7 MB/s eta 0:00:01
   -------------------------------------- - 15.5/16.1 MB 36.7 MB/s eta 0:00:01
   ---------------------------------------- 16.1/16.1 MB 35.7 MB/s  0:00:00
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)

  Attempting uninstall: python-dateutil

    Found existing installation: python-dateutil 4.0.0

    Uninstalling python-dateutil-4.0.0:

      Successfully uninstalled python-dateutil-4.0.0

   ----- ----------------------------------  2/15 [propcache]
   ------------- --------------------------  5/15 [fsspec]
   ------------- --------------------------  5/15 [fsspec]
   -------------------------- ------------- 10/15 

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow.compute as pc
import pyarrow.dataset as ds
import s3fs

YEAR = 2023
RELEASE = "v2026.10.0"
BASE = f"pudl.catalyst.coop/{RELEASE}"

s3 = s3fs.S3FileSystem(anon=True)

def read_year(table_name, columns):
    dataset = ds.dataset(
        f"{BASE}/{table_name}.parquet",
        filesystem=s3,
        format="parquet",
    )
    return dataset.to_table(
        columns=columns,
        filter=pc.year(ds.field("report_date")) == YEAR,
    ).to_pandas()


generators = read_year(
    "out_eia__yearly_generators",
    [
        "report_date", "plant_id_eia", "generator_id",
        "operational_status_code", "energy_source_code_1",
        "fuel_type_code_pudl", "capacity_mw",
    ],
)

plants = read_year(
    "out_eia__yearly_plants",
    [
        "report_date", "plant_id_eia", "plant_name_eia",
        "sector_id_eia", "sector_name_eia",
        "state", "latitude", "longitude",
    ],
)

print(f"Generator records: {len(generators):,}")
print(f"Plant records:     {len(plants):,}")
display(generators.head())
display(plants.head())

Generator records: 33,446
Plant records:     15,112


,report_date,plant_id_eia,generator_id,operational_status_code,energy_source_code_1,fuel_type_code_pudl,capacity_mw
0,2023-01-01,1,1,SB,DFO,oil,0.9
1,2023-01-01,1,2,OP,DFO,oil,0.9
2,2023-01-01,1,3,OP,DFO,oil,0.5
3,2023-01-01,1,5,RE,DFO,oil,0.7
4,2023-01-01,1,5.1,OP,DFO,oil,0.4


,report_date,plant_id_eia,plant_name_eia,sector_id_eia,sector_name_eia,state,latitude,longitude
0,2023-01-01,67912,Claverack Creek Solar,2,IPP Non-CHP,NY,42.271000,-73.750000
1,2023-01-01,67911,Catskill Solar,2,IPP Non-CHP,NY,42.270000,-73.748000
2,2023-01-01,67910,"Timbermill Wind, LLC",2,IPP Non-CHP,NC,36.151863,-76.601040
3,2023-01-01,67909,Sherco Solar I,1,Electric Utility,MN,45.407183,-93.936502
4,2023-01-01,67908,Richmond (NY),2,IPP Non-CHP,NY,42.792865,-77.526792


## Step 2.1 Check the records before combining them
This catches a consequential error early: if a generator appears twice, its capacity could be counted twice. The sector table also lets you see exactly which commercial and industrial categories your screen will retain.

In [2]:
plant_key = ["report_date", "plant_id_eia"]
generator_key = plant_key + ["generator_id"]

assert not generators.empty and not plants.empty, "No data returned for this year."
assert generators[generator_key].notna().all().all(), "Missing generator identifier."
assert plants[plant_key].notna().all().all(), "Missing plant identifier."
assert not generators.duplicated(generator_key).any(), "Duplicate generator-year."
assert not plants.duplicated(plant_key).any(), "Duplicate plant-year."

display(
    plants[["sector_id_eia", "sector_name_eia"]]
    .drop_duplicates()
    .sort_values("sector_id_eia")
)

display(generators["operational_status_code"].value_counts(dropna=False))

,sector_id_eia,sector_name_eia
3,1,Electric Utility
0,2,IPP Non-CHP
442,2,<NA>
259,2,NAICS-22 Non-Cogen
857,3,NAICS-22 Cogen
605,3,IPP CHP
121,4,Commercial Non-CHP
737,4,Commercial NAICS Non-Cogen
1397,4,<NA>
2881,5,Commercial NAICS Cogen


operational_status_code
OP      23765
RE       4379
SB       1586
CN        755
P         487
OS        482
U         478
V         442
L         358
T         277
OA        176
TS        131
IP        116
<NA>       11
OT          3
Name: count, dtype: int64[pyarrow]